In [2]:
import os
import yaml
import base64
import fitz  # PyMuPDF (페이지를 이미지로 변환하기 위해 필요)
import warnings; warnings.filterwarnings('ignore')
from dotenv import load_dotenv

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

In [3]:
load_dotenv(dotenv_path="../.env") 
with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

gemini_api_key = os.getenv(config['api_keys']['google_gemini_env_name'])
if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    print("✅ Gemini API 키 로드 완료")
else:
    print("❌ API 키 오류")

✅ Gemini API 키 로드 완료


In [4]:
pdf_path = "../data/raw/samsung_report.pdf"

print("⏳ 텍스트 벡터 DB를 구축/로드합니다... (Baseline과 동일한 방식)")
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(documents)

embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ 텍스트 기반 Vector DB 구축 완료!")

⏳ 텍스트 벡터 DB를 구축/로드합니다... (Baseline과 동일한 방식)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33174.34it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 텍스트 기반 Vector DB 구축 완료!


In [7]:
print("\n" + "="*50)
print("🧐 AI에게 어려운 질문을 던져봅시다 (하이브리드 테스트)")

query = "2024년 삼성전자의 당기순이익은 얼마인가요?"
print(f"질문: {query}\n")

# --- 1단계: 텍스트 검색으로 정답이 있을 확률이 가장 높은 '페이지 번호' 알아내기 ---
print("🔍 [Path A 작동] 텍스트 키워드로 정확한 페이지 위치를 찾는 중...")
retrieved_docs = retriever.invoke(query)

# 가장 유사도가 높은 첫 번째 문서의 메타데이터에서 페이지 번호 추출 (PyMuPDF는 0부터 시작함)
best_page_num = retrieved_docs[0].metadata.get('page', 0)
best_page_num = 295
print(f"🎯 텍스트 검색 결과: 정답은 {best_page_num + 1}페이지에 있을 확률이 높습니다!")

# --- 2단계: 알아낸 페이지 번호를 고화질 '이미지'로 변환하기 ---
print(f"📸 [Path B 작동] {best_page_num + 1}페이지의 원본 형태(레이아웃)를 이미지로 캡처합니다...")

doc = fitz.open(pdf_path)
page = doc.load_page(best_page_num)
# 해상도를 높이기 위해 2배 확대(zoom)하여 렌더링
zoom_matrix = fitz.Matrix(2.0, 2.0) 
pix = page.get_pixmap(matrix=zoom_matrix)

# 이미지를 Base64 문자열로 변환 (Gemini에게 보내기 위함)
img_bytes = pix.tobytes("png")
best_page_base64 = base64.b64encode(img_bytes).decode('utf-8')
print("✅ 이미지 캡처 및 변환 완료!")


🧐 AI에게 어려운 질문을 던져봅시다 (하이브리드 테스트)
질문: 2024년 삼성전자의 당기순이익은 얼마인가요?

🔍 [Path A 작동] 텍스트 키워드로 정확한 페이지 위치를 찾는 중...
🎯 텍스트 검색 결과: 정답은 296페이지에 있을 확률이 높습니다!
📸 [Path B 작동] 296페이지의 원본 형태(레이아웃)를 이미지로 캡처합니다...
✅ 이미지 캡처 및 변환 완료!


In [ ]:
# ==========================================
# 🧠 [최종 융합] VLM(Gemini 2.5 Flash)에게 텍스트 컨텍스트와 이미지를 동시 제공
# ==========================================
print("🧠 Gemini 2.5 Flash에게 '검색된 텍스트'와 '해당 페이지 이미지'를 모두 제공하여 교차 검증합니다...\n")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 추출했던 텍스트 내용(참고용)
text_context = retrieved_docs[0].page_content

message = HumanMessage(
    content=[
        {
            "type": "text", 
            "text": f"""당신은 꼼꼼한 금융 데이터 분석가입니다.
아래 제공된 [텍스트 컨텍스트]와 [문서 이미지]를 '교차 검증'하여 사용자의 [질문]에 답변해주세요.
숫자의 단위나 표의 행/열 구조는 반드시 [문서 이미지]를 기준으로 파악해야 합니다.

[텍스트 컨텍스트 (참고용)]
{text_context}

[질문]
{query}
"""
        },
        {
            "type": "image_url", 
            "image_url": {"url": f"data:image/jpeg;base64,{best_page_base64}"}
        }
    ]
)

response = llm.invoke([message])

print("=" * 50)
print("🤖 최종 AI 답변 (Dual-Path Hybrid RAG):")
print(response.content)
print("=" * 50)

🧠 Gemini 2.5 Flash에게 '검색된 텍스트'와 '해당 페이지 이미지'를 모두 제공하여 교차 검증합니다...

🤖 최종 AI 답변 (Dual-Path Hybrid RAG):
제공된 [문서 이미지]와 [텍스트 컨텍스트]를 교차 검증한 결과, 2024년은 '제56기'에 해당합니다.

[문서 이미지]의 '나. 영업실적' 표에서 '제56기'의 '당기순이익'은 **34,451,351 백만원**입니다.
